In [1]:
import pandas as pd
from darts.timeseries import TimeSeries
from darts.utils.timeseries_generation import datetime_attribute_timeseries
from darts.dataprocessing.transformers import Scaler
from darts.models import TFTModel
from darts.dataprocessing.transformers import StaticCovariatesTransformer
import numpy as np
import torch
import matplotlib.pyplot as plt
import joblib
import os

The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md


### Loss Logger

In [2]:
from pytorch_lightning.callbacks import Callback

class LossLogger(Callback):
    """
    A PyTorch Lightning callback to record training and validation losses 
    at the end of every epoch for custom plotting or analysis.
    """
    def __init__(self):
        super().__init__()
        self.train_losses = []
        self.val_losses = []

    def on_train_epoch_end(self, trainer, pl_module):
        # Retrieve train_loss from callback_metrics
        train_loss = trainer.callback_metrics.get("train_loss")
        
        if train_loss is not None:
            # detach() ensures we don't keep the computation graph in memory
            # cpu() ensures it works regardless of whether you're on GPU or CPU
            self.train_losses.append(float(train_loss.detach().cpu()))

    def on_validation_epoch_end(self, trainer, pl_module):
        # Retrieve val_loss from callback_metrics
        val_loss = trainer.callback_metrics.get("val_loss")
        
        if val_loss is not None:
            self.val_losses.append(float(val_loss.detach().cpu()))

### Add encoders

In [3]:
# Function to encode the year as a normalized value
def encode_year(idx):
  return (idx.year - 2000) / 50

def encode_days_in_month(index):
  return index.days_in_month.to_numpy().reshape(-1,1)

# Set up the add_encoders dictionary to specify how different time-related encoders and transformers should be applied
add_encoders = {
    'cyclic': {'past': ['month'], 'future': ['month']},
    'position': {'past': ['relative'], 'future': ['relative']},
    'custom': {
        'past': [encode_year, encode_days_in_month],
        'future': [encode_year, encode_days_in_month]
    },
    'transformer': Scaler()
}

### Actual sales : Jun'25 to Nov'25

In [4]:
file_path = r"C:\Users\G0004878\Desktop\TFT_Data\6_month_horizon\Final_results_over_different_iterations.xlsx"
excel_file = pd.ExcelFile(file_path)
excel_file.sheet_names

['Iteration 1',
 'Iteration 2',
 'Iteration 2 details',
 'Iteration 3',
 'Iteration 3 details',
 'Iteration 4',
 'Iteration 4 details',
 'Pivot',
 'Actual sales over last 4 years',
 'Sheet9']

In [5]:
actual_sales = pd.read_excel(file_path,sheet_name='Actual sales over last 4 years')
actual_sales = actual_sales.set_index('Month of sale')
actual_sales.index.name = 'MONTH_OF_SALE'
actual_sales=actual_sales.loc['2025-06-01':'2025-11-01',:]

In [6]:
actual_sales = actual_sales.drop(columns=["Name of month","MONTH_NUM"])


In [7]:
actual_sales.rename({"Actual sales":'ACTUAL_SALES'},inplace=True)

### Load the model

In [8]:
from darts.models import TFTModel

WORK_DIR = r"C:\Users\G0004878\Desktop\TFT_Data\6_month_horizon\darts_logs"

loaded_model = TFTModel.load_from_checkpoint(work_dir=WORK_DIR,model_name='tft_engineered_features_6_month_with_modifications_2026-06-15_16_27_54',best=True,map_location="cpu")
print("Model loaded successfully with all dimensions intact!")

Model loaded successfully with all dimensions intact!


### Data preparation

In [9]:
#Read the data 
pandas_df = pd.read_csv(r"C:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Pipeline\Bug fixing\Updated input data_post feature engineering\Series_A_data.csv",index_col = ['MONTH_OF_SALE'],parse_dates=True)

In [10]:
#Extracting type of columns according to the datatypes
# 1. Targets/Metrics (The numbers we want to predict)
target_cols = pandas_df.select_dtypes(include=['number']).columns.tolist()
target_cols.append('PARENT_DEALER_CODE_MODEL_FAMILY')

# 2. Time Dimension
time_cols = pandas_df.select_dtypes(include=['datetime', 'datetime64']).columns.tolist()

# 3. Static/Categorical Covariates (The identifiers)
static_cols = pandas_df.select_dtypes(exclude=['number', 'datetime', 'datetime64']).columns.tolist()
static_cols.append('PARENT_DEALER_CODE')

print(f"Targets: {target_cols}")
print(f"Time Column: {time_cols}")
print(f"Static Identifiers: {static_cols}")

Targets: ['PARENT_DEALER_CODE', 'NET_SALES', 'MARRIAGE_DAYS', 'FESTIVE_PHASE_I', 'FESTIVE_PHASE_II', 'FESTIVE_PHASE_III', 'PITRU_PAKSH', 'PROP_FESTIVE_PHASE_I', 'PROP_FESTIVE_PHASE_II', 'PROP_FESTIVE_PHASE_III', 'PROP_PITRU_PAKSH', 'TOTAL_FESTIVAL_DAYS', 'LAST_YEAR_CONTRIBUTION', 'PARENT_DEALER_CODE_MODEL_FAMILY']
Time Column: []
Static Identifiers: ['MODEL_FAMILY', 'PARENT_DEALER_CODE_MODEL_FAMILY', 'BRAKE_TYPE', 'IGNITION_TYPE', 'WHEEL_TYPE', 'COLOUR', 'DEALER_CITY', 'X_CITY_CATEGORY', 'ZONAL_OFFICE_NAME', 'PARENT_DEALER_CODE']


In [11]:
#Separating the covariates
target_col = ["NET_SALES"]

#future covariates
future_covariates = [i for i in target_cols if i not in ['NET_SALES','PARENT_DEALER_CODE']]

#actual_static_cols
actual_static_cols = [i for i in static_cols if i!='PARENT_DEALER_CODE_MODEL_FAMILY']

In [12]:
static_covariates = actual_static_cols.copy()

static_covariates

['MODEL_FAMILY',
 'BRAKE_TYPE',
 'IGNITION_TYPE',
 'WHEEL_TYPE',
 'COLOUR',
 'DEALER_CITY',
 'X_CITY_CATEGORY',
 'ZONAL_OFFICE_NAME',
 'PARENT_DEALER_CODE']

In [13]:
pandas_df=pandas_df.reset_index().sort_values(by=["PARENT_DEALER_CODE_MODEL_FAMILY","MONTH_OF_SALE"]).set_index("MONTH_OF_SALE")

pandas_df_with_target_and_static_covariates = pandas_df.loc[:,['PARENT_DEALER_CODE_MODEL_FAMILY','NET_SALES']+static_covariates]

pandas_df_with_future_covariates = pandas_df.loc[:,future_covariates]

darts_df_with_static_covariates = TimeSeries.from_group_dataframe(df=pandas_df_with_target_and_static_covariates,
                                                                  group_cols=["PARENT_DEALER_CODE_MODEL_FAMILY"],
                                                                  static_cols=static_covariates,value_cols=["NET_SALES"],freq='MS')


In [14]:
try:
    future_covariates.remove('PARENT_DEALER_CODE_MODEL_FAMILY')
except:
    pass

darts_df_with_future_covariates = TimeSeries.from_group_dataframe(df = pandas_df_with_future_covariates,
                                    group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
                                    freq = 'MS',
                                    value_cols = future_covariates
                                    )

### Load the scalers

In [15]:
scaler_dir = r"C:\Users\G0004878\Desktop\TFT_Data\6_month_horizon\Huber loss\Saved_scaled_objects_huber_loss"
target_scaler = joblib.load(os.path.join(scaler_dir,'target_scaler.pkl'))

future_covariates_scaler = joblib.load(os.path.join(scaler_dir,'future_covariates_scaler.pkl'))

transformer = joblib.load(os.path.join(scaler_dir,'static_transformer.pkl'))

In [16]:
full_target_list = [
    ts.slice(pd.Timestamp('2023-04-01'), pd.Timestamp('2025-11-01'))
    for ts in darts_df_with_static_covariates
]

scaled_full_target = target_scaler.transform(full_target_list)
scaled_full_target = transformer.transform(scaled_full_target)


full_futcov_list = [
    ts.slice(pd.Timestamp('2023-04-01'), pd.Timestamp('2025-11-01'))
    for ts in darts_df_with_future_covariates
]

scaled_full_futcov = future_covariates_scaler.transform(full_futcov_list)

### Backtesting using Mean prediction of quantile loss

In [17]:
#Iteration 1 is Mean Prediction of Quantile Loss
backtest_preds_iteration1 = loaded_model.historical_forecasts(series=scaled_full_target,future_covariates=scaled_full_futcov,forecast_horizon=6,start=pd.Timestamp('2025-06-01'),stride=6,retrain=False,last_points_only=False,overlap_end=False,num_samples=1,verbose=True)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX 2000 Ada Generation') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

Generating TimeSeries:   0%|          | 0/37719 [00:00<?, ?it/s]

In [18]:
backtest_actual_scale_iteration1 = target_scaler.inverse_transform(backtest_preds_iteration1)

### Load the Huber Loss model

In [19]:
WORK_DIR = r"C:\Users\G0004878\Desktop\TFT_Data\6_month_horizon\Huber loss\darts_logs"

loaded_model_iteration2 = TFTModel.load_from_checkpoint(work_dir=WORK_DIR,model_name='tft_engineered_features_6_month_horizon_with_modifications_huber_loss_2026-06-15_22_49_16',best=True,map_location="cpu")
print("Model loaded successfully with all dimensions intact!")

Model loaded successfully with all dimensions intact!


### Backtesting using Huber loss

In [20]:
#Iteration 2 is Huber Loss
backtest_preds_iteration2 = loaded_model_iteration2.historical_forecasts(series=scaled_full_target,future_covariates=scaled_full_futcov,forecast_horizon=6,start=pd.Timestamp('2025-06-01'),stride=6,retrain=False,last_points_only=False,overlap_end=False,num_samples=1,verbose=True)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

Generating TimeSeries:   0%|          | 0/37719 [00:00<?, ?it/s]

In [21]:
backtest_actual_scale_iteration2 = target_scaler.inverse_transform(backtest_preds_iteration2)

In [30]:
# Build output DataFrame for Apr'26 to Jun'26 predictions
records = []

for i,(forecast, source_series) in enumerate(zip(backtest_actual_scale_iteration1,darts_df_with_static_covariates)):
    # val_list retains original static covariates — use it as the label source
    series_name = source_series.static_covariates['PARENT_DEALER_CODE_MODEL_FAMILY'].values[0]

    months          = forecast[i].time_index
    forecast_values = forecast[i].values().flatten()

    for month, pred in zip(months, forecast_values):
        records.append({
            'MONTH_OF_SALE'                   : month,
            'PARENT_DEALER_CODE_MODEL_FAMILY'  : series_name,
            'PREDICTED_SALES'                  : round(float(pred), 2)
        })

df_final_output = pd.DataFrame(records)
df_final_output['MONTH_OF_SALE'] = pd.to_datetime(df_final_output['MONTH_OF_SALE']).dt.strftime('%Y-%m-%d')
df_final_output = df_final_output.sort_values(['PARENT_DEALER_CODE_MODEL_FAMILY', 'MONTH_OF_SALE']).reset_index(drop=True)

print(f'Output shape : {df_final_output.shape}')
print(f'Months       : {df_final_output["MONTH_OF_SALE"].unique()}')
print(f'Series count : {df_final_output["PARENT_DEALER_CODE_MODEL_FAMILY"].nunique()}')
df_final_output.head(10)

IndexError: list index out of range